# Round 07 / Features and the no-fit audit
Run each cell individually. All feature matrices and candidate proposals are committed **before** the audit stage reads training targets. The target-aware audit uses the same training-only pilot and censors targets before the earliest validation cutoff. These are support and candidate-oracle measurements, not new ranker scores.

In [1]:
from pathlib import Path
import json, subprocess, sys
ROOT=Path.home()/"otto_feature_round07"
assert (ROOT/"launch.py").is_file(), "Run install_round07.py first; do not reinstall the environment"
INTERFACE=Path("/opt/conda/bin/python")
assert INTERFACE.is_file(), "Existing notebook interface missing; stop"
def stage(name):
    return subprocess.run([str(INTERFACE),"-u",str(ROOT/"launch.py"),name],cwd=ROOT,check=True)
def gate(filename, expected):
    row=json.loads((ROOT/"outputs"/filename).read_text())
    assert row["status"]==expected, "Prior stage incomplete; stop and bundle evidence"
    print(expected)
print("KERNEL_READY", "interface:",sys.executable,"ML:",Path.home()/"otto-recommender-system/.venv/bin/python")

KERNEL_READY interface: /opt/conda/bin/python ML: /home/sagemaker-user/otto-recommender-system/.venv/bin/python


## Require the historical-session checkpoint

In [2]:
gate("history.json","ROUND07_HISTORY_READY")

ROUND07_HISTORY_READY


## Build 12 session-context features and the 12-feature last-anchor comparison

In [3]:
stage("features")

RUNNING features; process cap 140s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round07/outputs/features.log
{"completed": 32, "event": "feature_chunk_committed", "stage": "session_neighbor_features", "total": 256, "utc": "2026-09-12T22:22:17.793411+00:00"}
{"completed": 64, "event": "feature_chunk_committed", "stage": "session_neighbor_features", "total": 256, "utc": "2026-09-12T22:22:18.055036+00:00"}
{"completed": 96, "event": "feature_chunk_committed", "stage": "session_neighbor_features", "total": 256, "utc": "2026-09-12T22:22:18.263870+00:00"}
{"completed": 128, "event": "feature_chunk_committed", "stage": "session_neighbor_features", "total": 256, "utc": "2026-09-12T22:22:18.506785+00:00"}
{"completed": 160, "event": "feature_chunk_committed", "stage": "session_neighbor_features", "total": 256, "utc": "2026-09-12T22:22:18.740652+00:00"}
{"completed": 192, "event": "feature_chunk_committed", "stage": "session_neighbor_features", "total": 256, "utc": "2026-09-1

CompletedProcess(args=['/opt/conda/bin/python', '-u', '/home/sagemaker-user/otto_feature_round07/launch.py', 'features'], returncode=0)

## Audit support, new candidates and redundancy; generate eight charts

In [4]:
stage("audit")

RUNNING audit; process cap 110s. No AWS resource changes. Log: /home/sagemaker-user/otto_feature_round07/outputs/audit.log
{"decision": "REVIEW_FOR_MATCHED_RANKER_SCREEN_NOT_PROMOTION", "elapsed_seconds": 0.5044480740002655, "event": "phase_complete", "new_model_fits": 0, "peak_rss_mib": 209.9140625, "phase": "audit", "status": "ROUND07_AUDIT_READY", "utc": "2026-09-12T22:22:20.174528+00:00"}
RESULT: ROUND07_AUDIT_READY
FINISHED audit: 0.9 seconds
RETURN_FILE: /home/sagemaker-user/otto_feature_round07/otto_round07_return.zip


CompletedProcess(args=['/opt/conda/bin/python', '-u', '/home/sagemaker-user/otto_feature_round07/launch.py', 'audit'], returncode=0)

## Save and inspect the decision

In [5]:
gate("result.json","ROUND07_AUDIT_READY")
result=json.loads((ROOT/"outputs/result.json").read_text())
print(json.dumps({"decision":result["decision"],"new_model_fits":result["new_model_fits"],"baseline_candidate_oracle":result["baseline_candidate_oracle"],"arms":result["arms"]},indent=2))
print("Save this notebook. Open 03_saved_results.ipynb, then return otto_round07_return.zip.")

ROUND07_AUDIT_READY
{
  "decision": "REVIEW_FOR_MATCHED_RANKER_SCREEN_NOT_PROMOTION",
  "new_model_fits": 0,
  "baseline_candidate_oracle": {
    "denominators": [
      236,
      70,
      33
    ],
    "hits": [
      158,
      37,
      18
    ],
    "recall": {
      "carts": 0.5285714285714286,
      "clicks": 0.6694915254237288,
      "orders": 0.5454545454545454
    },
    "weighted_oracle_at20": 0.5527933083865286
  },
  "arms": {
    "last_anchor": {
      "expanded_candidate_oracle": {
        "denominators": [
          236,
          70,
          33
        ],
        "hits": [
          164,
          38,
          18
        ],
        "recall": {
          "carts": 0.5428571428571428,
          "clicks": 0.6949152542372882,
          "orders": 0.5454545454545454
        },
        "weighted_oracle_at20": 0.5596213955535989
      },
      "mean_new_candidates_by_objective": [
        60.1015625,
        35.23046875,
        14.94921875
      ],
      "net_capped_hits":